# 🌍 Land aus Koordinaten (ohne API-Key)
Dieses Notebook zeigt, wie man mit **Lat/Lon** das Land abfragen kann – ohne API-Key.

Zwei Dienste werden verglichen:
- **BigDataCloud** – einfach, kein Header nötig
- **Nominatim (OpenStreetMap)** – detaillierter, User-Agent empfohlen

In [11]:
%pip install requests pandas --quiet

Note: you may need to restart the kernel to use updated packages.


In [12]:
import requests
import json
import time
import pandas as pd

## 1️⃣ BigDataCloud API
Einfachster Ansatz – kein Key, kein Header erforderlich.

In [13]:
def get_country_bigdatacloud(lat: float, lon: float, language: str = "de") -> dict:
    """
    Gibt Länderinformationen via BigDataCloud zurück.
    Kein API-Key erforderlich.
    """
    url = "https://api.bigdatacloud.net/data/reverse-geocode-client"
    params = {
        "latitude": lat,
        "longitude": lon,
        "localityLanguage": language
    }
    response = requests.get(url, params=params)
    response.raise_for_status()
    data = response.json()
    return {
        "land": data.get("countryName"),
        "land_code": data.get("countryCode"),
        "region": data.get("principalSubdivision"),
        "ort": data.get("city") or data.get("locality"),
        "quelle": "BigDataCloud"
    }

In [14]:
# Beispiel: Köln, Deutschland
lat, lon = 50.9333, 6.9500
ergebnis = get_country_bigdatacloud(lat, lon)
print(json.dumps(ergebnis, indent=2, ensure_ascii=False))

{
  "land": "Deutschland",
  "land_code": "DE",
  "region": "Nordrhein-Westfalen",
  "ort": "Köln",
  "quelle": "BigDataCloud"
}


## 2️⃣ Nominatim API (OpenStreetMap)
Detailliertere Daten. **Wichtig:** User-Agent Header setzen und max. 1 Request/Sekunde.

In [18]:
def get_country_nominatim(lat: float, lon: float) -> dict:
    """
    Gibt Länderinformationen via Nominatim (OpenStreetMap) zurück.
    Kein API-Key erforderlich, aber User-Agent wird erwartet.
    Bitte max. 1 Request pro Sekunde (Fair Use Policy).
    """
    url = "https://nominatim.openstreetmap.org/reverse"
    headers = {
        "User-Agent": "MeinPythonNotebook/1.0 (kontakt@beispiel.de)"  # Bitte anpassen!
    }
    params = {
        "lat": lat,
        "lon": lon,
        "format": "json",
        "accept-language": "en"
    }
    response = requests.get(url, params=params, headers=headers)
    response.raise_for_status()
    data = response.json()
    adresse = data.get("address", {})
    return {
        "land": adresse.get("country"),
        "land_code": adresse.get("country_code", "").upper(),
        "bundesland": adresse.get("state"),
        "ort": adresse.get("city") or adresse.get("town") or adresse.get("village"),
        "display_name": data.get("display_name"),
        "quelle": "Nominatim (OSM)"
    }

In [19]:
# Beispiel: Köln, Deutschland
ergebnis_osm = get_country_nominatim(lat, lon)
print(json.dumps(ergebnis_osm, indent=2, ensure_ascii=False))

{
  "land": "Germany",
  "land_code": "DE",
  "bundesland": "North Rhine-Westphalia",
  "ort": "Cologne",
  "display_name": "98, Agrippastraße, Cäcilienviertel, Altstadt-Süd, Innenstadt, Cologne, North Rhine-Westphalia, 50676, Germany",
  "quelle": "Nominatim (OSM)"
}


In [24]:
def fetch_location_name(lat: float, lon: float) -> str:
    """Nominatim: human-readable location name for given lat/lon."""
    try:
        url = "https://nominatim.openstreetmap.org/reverse"
        headers = {
            "User-Agent": "MeinPythonNotebook/1.0 (kontakt@beispiel.de)"  # Bitte anpassen!
        }
        params = {
            "lat": lat,
            "lon": lon,
            "format": "json",
            "accept-language": "en"
        }
        response = requests.get(url, params=params, headers=headers)
        response.raise_for_status()
        data = response.json()
        adresse = data.get("address", {})
        return {
            "land": adresse.get("country"),
            "bundesland": adresse.get("state"),
            "ort": adresse.get("city") or adresse.get("town") or adresse.get("village"),
        }
    except Exception as e:
        print(f"Nominatim fetch failed: {e}", flush=True)
        return ""
    
fetch_location_name(500.9333, 6.9500)

{'land': None, 'bundesland': None, 'ort': None}

## 3️⃣ Vergleich beider APIs für mehrere Koordinaten

In [17]:
koordinaten = [
    {"name": "Köln",        "lat": 50.9333, "lon": 6.9500},
    {"name": "Paris",       "lat": 48.8566, "lon": 2.3522},
    {"name": "New York",    "lat": 40.7128, "lon": -74.0060},
    {"name": "Tokio",       "lat": 35.6762, "lon": 139.6503},
    {"name": "Sydney",      "lat": -33.8688, "lon": 151.2093},
]

ergebnisse = []

for ort in koordinaten:
    print(f"Abfrage: {ort['name']} ({ort['lat']}, {ort['lon']})...")
    
    bdc = get_country_bigdatacloud(ort["lat"], ort["lon"])
    time.sleep(1)  # Nominatim Rate-Limit einhalten
    osm = get_country_nominatim(ort["lat"], ort["lon"])
    
    ergebnisse.append({
        "Ort": ort["name"],
        "Lat": ort["lat"],
        "Lon": ort["lon"],
        "Land (BigDataCloud)": bdc["land"],
        "Code (BDC)": bdc["land_code"],
        "Land (Nominatim)": osm["land"],
        "Code (OSM)": osm["land_code"],
    })
    time.sleep(1)

df = pd.DataFrame(ergebnisse)
df

Abfrage: Köln (50.9333, 6.95)...
Abfrage: Paris (48.8566, 2.3522)...
Abfrage: New York (40.7128, -74.006)...
Abfrage: Tokio (35.6762, 139.6503)...
Abfrage: Sydney (-33.8688, 151.2093)...


,Ort,Lat,Lon,Land (BigDataCloud),Code (BDC),Land (Nominatim),Code (OSM)
0,Köln,50.9333,6.9500,Deutschland,DE,Deutschland,DE
1,Paris,48.8566,2.3522,Frankreich,FR,Frankreich,FR
2,New York,40.7128,-74.0060,Vereinigte Staaten,US,Vereinigte Staaten von Amerika,US
3,Tokio,35.6762,139.6503,Japan,JP,Japan,JP
4,Sydney,-33.8688,151.2093,Australien,AU,Australien,AU


## 4️⃣ Eigene Koordinaten testen

In [ ]:
# ✏️ Hier eigene Koordinaten eintragen:
mein_lat = 52.5200   # Berlin
mein_lon = 13.4050

print("=== BigDataCloud ===")
print(json.dumps(get_country_bigdatacloud(mein_lat, mein_lon), indent=2, ensure_ascii=False))

time.sleep(1)

print("\n=== Nominatim (OSM) ===")
print(json.dumps(get_country_nominatim(mein_lat, mein_lon), indent=2, ensure_ascii=False))

---
## 📋 Zusammenfassung

| Dienst | API-Key | Rate-Limit | Besonderheiten |
|---|---|---|---|
| **BigDataCloud** | ❌ nicht nötig | großzügig | Sehr einfach, gut für Client-Apps |
| **Nominatim (OSM)** | ❌ nicht nötig | 1 Req/Sek | Detailreicher, User-Agent setzen |